# Replicating Guess et al. (2023): the Chronological Feed experiment

**Paper:** Guess, A. M., et al. (2023). *How do social media feed algorithms affect attitudes and behavior in an election campaign?* **Science**, 381, 398. DOI: [10.1126/science.abp9364](https://doi.org/10.1126/science.abp9364)

**What this notebook is.** A *methodological* replication, not a copy. The original data is restricted-access (Meta's US 2020 Facebook & Instagram Election Study, released only to vetted researchers through ICPSR/SOMAR). We cannot touch it. So we (1) reconstruct the study's design and analysis pipeline, (2) generate a **synthetic dataset** with the same moving parts and with the paper's reported effect sizes *planted as ground truth*, and (3) show that the pipeline recovers them and reproduces the paper's qualitative pattern.

**The finding we are reproducing (the headline).** Switching users from the algorithmic feed to a reverse-chronological feed *dramatically* changed **what they saw** (more political content, more from untrustworthy and cross-cutting sources) and **cut their on-platform engagement** — yet had **no measurable effect on political attitudes**: affective/issue polarization, knowledge, and self-reported participation were all unchanged over the ~3-month study.

**A caveat the authors and Science have since flagged.** Facebook's ranking algorithm was *in flux* during the study window (Sept–Dec 2020) because of "break-glass" election measures. Science attached an [editorial (Thorp & Vinson, 2024)](https://doi.org/10.1126/science.adt2983) noting the control-group "algorithmic feed" was not a fixed object. We carry this forward as a documented limitation (see `replication_log.md`).

> **How to read the figures:** a blue tick marks the *true effect we planted*; a red dot/line means the estimate is statistically significant after multiple-comparison correction; grey means not significant.

## 0. Setup
Everything below is plain `numpy` / `pandas` / `matplotlib`, plus `scikit-learn` for the lasso covariate-selection step (preinstalled in Colab). Nothing here is hidden behind a heavy regression library — the robust standard errors are built by hand so you can see exactly what they are.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 2020          # the paper seeds its lasso with 2020; we reuse it everywhere
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 1. Configuration: what we measure and what we plant

Two kinds of outcome appear in the paper:

* **First-stage / "RQ" outcomes** — what the feed *shows you*, measured in **percentage points of the feed**. The paper finds big, significant effects here. These are the intervention working as intended.
* **Attitude / behaviour outcomes** — the downstream things, measured in **standard-deviation (z-score) units**. The paper finds *almost nothing* here. This is the null result.

`true_effect` is the population average treatment effect (PATE) we plant, taken from the paper's Facebook tables (SM Tables S2, S4, S5). Because we plant them, success means *recovering* them — not matching Meta's exact numbers, which is impossible with synthetic data.

In [ ]:
# Facebook "study-completes" (the analysed sample), SM section S1.1.
N_CONTROL_FB, N_TREAT_FB = 16_159, 7_232
N_CONTROL_IG, N_TREAT_IG = 12_514, 8_800        # Instagram, for an optional second run
NONCOMPLIANCE_RATE_FB = 0.119                    # ~11.9% of treated FB users (web bug)

# --- First stage (RQ): effect on feed composition, in PERCENTAGE POINTS ---
# true_effect = PATE from SM Table S4 (Facebook).
RQ_OUTCOMES = {
    "pct_political_content": {"label": "% of feed that is political content",  "control_mean": 12.0, "control_sd": 9.0,  "true_effect": +1.68},
    "pct_cross_cutting":     {"label": "% of feed from cross-cutting sources", "control_mean": 20.0, "control_sd": 12.0, "true_effect": -2.48},
    "pct_political_news":    {"label": "% of feed that is political news",     "control_mean": 6.0,  "control_sd": 6.0,  "true_effect": +1.82},
    "pct_untrustworthy":     {"label": "% of feed from untrustworthy sources", "control_mean": 1.3,  "control_sd": 2.0,  "true_effect": +1.59},
    "pct_uncivil":           {"label": "% of feed that is uncivil",            "control_mean": 6.0,  "control_sd": 4.0,  "true_effect": -1.33},
    "pct_slur_words":        {"label": "% of feed with slur words",            "control_mean": 0.10, "control_sd": 0.20, "true_effect": -0.02},
}

# --- Primary hypotheses H1-H3, in SD UNITS. true_effect = PATE from SM Table S2. ---
# Note: the only real effect is H3c (on-platform engagement) -- a BEHAVIOUR, not an attitude.
PRIMARY_OUTCOMES = {
    "affective_polarization": {"label": "H1a: Affective polarization",       "true_effect":  0.000},
    "issue_polarization":     {"label": "H1b: Issue polarization",           "true_effect":  0.000},
    "election_knowledge":     {"label": "H2a: Election knowledge",           "true_effect":  0.000},
    "news_knowledge":         {"label": "H2b: News knowledge",              "true_effect": -0.025},
    "self_participation":     {"label": "H3a: Self-reported participation",  "true_effect": -0.025},
    "self_turnout":           {"label": "H3b: Self-reported turnout",        "true_effect":  0.000},
    "onplatform_engagement":  {"label": "H3c: On-platform pol. engagement",  "true_effect": -0.118},
}

# --- A subset of secondary hypotheses, in SD UNITS. true_effect = PATE from SM Table S5. ---
SECONDARY_OUTCOMES = {
    "factual_discernment":     {"label": "SH1a: Knowledge / information",       "true_effect":  0.000},
    "trust_media":             {"label": "SH2a: Trust in media (excl. social)", "true_effect":  0.000},
    "trust_social_media":      {"label": "SH2b: Trust in social media info",    "true_effect": +0.041},
    "confidence_institutions": {"label": "SH2c: Confidence in institutions",    "true_effect":  0.000},
    "perceived_polarization":  {"label": "SH3a: Perceived polarization",        "true_effect":  0.000},
    "partisan_news_clicks":    {"label": "SH3b: Partisan news clicks",          "true_effect": +0.107},
    "political_efficacy":      {"label": "SH4: Political efficacy",             "true_effect":  0.000},
    "belief_election_legit":   {"label": "SH6: Belief in election legitimacy",  "true_effect":  0.000},
}

ALL_STD_OUTCOMES = {**PRIMARY_OUTCOMES, **SECONDARY_OUTCOMES}
print(f"{len(RQ_OUTCOMES)} first-stage, {len(PRIMARY_OUTCOMES)} primary, {len(SECONDARY_OUTCOMES)} secondary outcomes")

## 2. Simulate the synthetic dataset

We build participants with the same structure as the real study:

1. **Pre-treatment covariates** — demographics, partisanship, baseline news use, baseline outcomes.
2. **Block randomisation** into Control vs Chronological Feed (within strata, so the arms are balanced).
3. **Imperfect compliance** — ~12% of treated Facebook users kept the algorithmic feed (a web bug fixed mid-study).
4. **Survey weights** — with enough spread that the *weighted* PATE has a realistic "design effect" (variance inflation ~3x) over the *unweighted* SATE. This is why, in the paper and here, PATE standard errors are larger than SATE ones.
5. **Outcomes** — built so the planted treatment effect is exactly what we put in `config`, on top of covariate-driven signal and noise.

In [ ]:
def draw_covariates(rng, n):
    df = pd.DataFrame(index=range(n))
    df["age_group"] = rng.choice(["18-29","30-44","45-65","65+"], n, p=[.22,.30,.33,.15])
    df["gender"]    = rng.choice(["Female","Male","Other"], n, p=[.53,.45,.02])
    df["race"]      = rng.choice(["White_NH","Black_NH","Hispanic","AAPI","Other"], n, p=[.62,.12,.16,.06,.04])
    df["party_id"]  = rng.choice(["Democrat","Republican","Independent"], n, p=[.40,.35,.25])
    party_centre = df["party_id"].map({"Democrat":3.0,"Republican":5.2,"Independent":4.1})
    df["ideology"] = np.clip(rng.normal(party_centre, 1.0), 1, 7)     # 1=v.liberal .. 7=v.conservative
    df["college_degree"]     = rng.binomial(1, 0.36, n)
    df["political_interest"] = rng.integers(1, 5, n)
    df["turnout_2016"]       = rng.binomial(1, 0.70, n)
    for ch in ["news_tv","news_cable","news_online","news_social","news_paper"]:
        df[ch] = np.clip(rng.beta(2, 5, n), 0, 1)
    df["pol_participation_pre"] = rng.integers(0, 7, n)
    df["digital_literacy"]      = np.clip(rng.normal(0, 1, n), -3, 3)
    return df

def assign_treatment_blockwise(rng, df, n_treat, n_control):
    df = df.copy()
    df["stratum"] = df["age_group"].astype(str) + " | " + df["party_id"].astype(str)
    share = n_treat / (n_treat + n_control)
    T = np.zeros(len(df), dtype=int)
    for _, idx in df.groupby("stratum").groups.items():       # randomise WITHIN each block
        idx = np.array(list(idx)); k = int(round(len(idx) * share))
        T[rng.choice(idx, size=k, replace=False)] = 1
    df["treatment"] = T
    return df

def draw_compliance(rng, df, nc_rate):
    df = df.copy(); n = len(df); pct = np.zeros(n); is_t = df["treatment"].values == 1
    pct[is_t]  = np.clip(rng.normal(0.97, 0.03, is_t.sum()), 0, 1)        # compliers
    t_idx = np.where(is_t)[0]; nc = rng.choice(t_idx, int(round(len(t_idx)*nc_rate)), replace=False)
    pct[nc]    = np.clip(rng.normal(0.35, 0.20, len(nc)), 0, 1)           # web-bug non-compliers
    pct[~is_t] = np.clip(rng.normal(0.01, 0.01, (~is_t).sum()), 0, 1)     # control
    df["pct_views_chrono"] = pct
    df["complier"] = (pct >= 0.80).astype(int)
    return df

def draw_weights(rng, df):
    # Wide spread -> design effect ~3, so PATE SEs > SATE SEs (as in the paper).
    df = df.copy(); base = np.ones(len(df))
    base *= df["age_group"].map({"18-29":2.2,"30-44":1.1,"45-65":0.7,"65+":0.8}).values
    base *= df["race"].map({"White_NH":0.75,"Black_NH":1.5,"Hispanic":1.7,"AAPI":1.3,"Other":1.1}).values
    base *= np.where(df["college_degree"].values == 1, 0.7, 1.35)
    base *= rng.lognormal(0.0, 0.85, len(df))
    df["weight"] = base / base.mean()
    return df

def covariate_signal(df, strength=0.45):
    x = (0.5*(df["ideology"].values-4) + 0.4*(df["political_interest"].values-2.5)
         + 0.3*(df["pol_participation_pre"].values-3) + 0.6*df["digital_literacy"].values
         + 0.4*df["turnout_2016"].values)
    return strength * (x - x.mean()) / x.std()

def add_rq_outcomes(df, rng):
    df = df.copy(); sig = covariate_signal(df)
    for name, s in RQ_OUTCOMES.items():
        y = (s["control_mean"] + s["true_effect"]*df["treatment"].values
             + s["control_sd"]*0.3*sig + rng.normal(0, s["control_sd"], len(df)))
        df[name] = np.clip(y, 0, 100)
    return df

def add_standardized_outcomes(df, rng):
    df = df.copy()
    for name, s in ALL_STD_OUTCOMES.items():
        trait = covariate_signal(df) + rng.normal(0, 0.9, len(df))
        pre   = trait + rng.normal(0, 0.6, len(df))                     # Wave 1/2 measure
        post  = 0.65*trait + s["true_effect"]*df["treatment"].values + rng.normal(0, 0.75, len(df))
        df[name + "_pre"] = (pre - pre.mean()) / pre.std()
        df[name]          = (post - post.mean()) / post.std()
    return df

def simulate_platform(platform, n_treat, n_control, nc_rate, seed):
    rng = np.random.default_rng(seed); n = n_treat + n_control
    df = draw_covariates(rng, n)
    df = assign_treatment_blockwise(rng, df, n_treat, n_control)
    df = draw_compliance(rng, df, nc_rate)
    df = draw_weights(rng, df)
    df = add_rq_outcomes(df, rng)
    df = add_standardized_outcomes(df, rng)
    df.insert(0, "platform", platform)
    return df

def simulate_all(seed=RANDOM_SEED):
    fb = simulate_platform("facebook",  N_TREAT_FB, N_CONTROL_FB, NONCOMPLIANCE_RATE_FB, seed)
    ig = simulate_platform("instagram", N_TREAT_IG, N_CONTROL_IG, 0.0, seed + 1)
    for col in ["pct_cross_cutting", "pct_political_news"]:    # FB-only classifiers
        ig[col] = np.nan
    return pd.concat([fb, ig], ignore_index=True)

data = simulate_all()
fb = data[data["platform"] == "facebook"].copy()
print("Sample sizes by arm:")
print(data.groupby(["platform","treatment"]).size())

### 2a. Sanity checks: balance and compliance
Randomisation should make the arms look alike on pre-treatment covariates, and the manipulation should show up in `pct_views_chrono`.

In [ ]:
# Covariate balance: treated vs control means should be close (randomisation worked).
bal_cols = ["ideology","political_interest","college_degree","turnout_2016","digital_literacy"]
balance = fb.groupby("treatment")[bal_cols].mean().T
balance.columns = ["control","treated"]
balance["abs_diff"] = (balance["treated"] - balance["control"]).abs()
print("Covariate balance (Facebook):"); print(balance.round(3))

print("\nManipulation check -- share of feed views in chronological order:")
print(fb.groupby("treatment")["pct_views_chrono"].mean().round(3))
print(f"\nWeight design effect = {1 + (fb['weight'].std()/fb['weight'].mean())**2:.2f}  (>1 means PATE noisier than SATE)")

## 3. The estimator

The paper's "saturated" specification is a **Lin (2013) regression** with **lasso-selected covariates** and **stratum dummies**, using **HC2 robust standard errors**, reporting a **weighted PATE** and an **unweighted SATE**, with p-values corrected by the **sharpened two-stage FDR** procedure (Benjamini–Krieger–Yekutieli 2006). Four ideas, explained as they appear.

### 3a. Lasso covariate selection
Ordinary regression keeps every control. **Lasso** adds a penalty that shrinks weak coefficients to exactly zero, so the data choose the controls. We run lasso *without* treatment (only to pick controls), then estimate the effect by OLS afterwards — "post-lasso" — which keeps the treatment estimate unbiased while soaking up variance.

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

def build_covariate_matrix(df, pre_col=None):
    numeric = ["ideology","political_interest","turnout_2016","news_tv","news_cable",
               "news_online","news_social","news_paper","pol_participation_pre",
               "digital_literacy","college_degree"]
    factors = ["age_group","gender","race","party_id"]
    X = df[numeric].copy()
    X = pd.concat([X, pd.get_dummies(df[factors], drop_first=True)], axis=1)  # drop_first avoids the dummy trap
    if pre_col is not None and pre_col in df.columns:
        X[pre_col] = df[pre_col].values
    return X.astype(float)

def select_covariates(X, y, seed=RANDOM_SEED):
    Xs = StandardScaler().fit_transform(X.values)
    lasso = LassoCV(cv=10, random_state=seed, max_iter=5000).fit(Xs, y)
    return X.columns[np.abs(lasso.coef_) > 1e-8].tolist()

### 3b. Robust standard errors, built by hand
A robust standard error is a "sandwich": `bread · meat · bread`, where `bread = (X'WX)^-1`. The variants differ in how they adjust the squared residuals that form the meat. **HC2** divides each squared residual by `(1 - hᵢ)`, where `hᵢ` is the observation's leverage; **HC1** applies a single `n/(n-k)` correction. The paper uses HC2 for OLS; for the weighted PATE we use HC1, the standard robust analogue for weighted least squares.

In [ ]:
def ols_robust(X, y, w=None, hc="HC2"):
    X = np.asarray(X, float); y = np.asarray(y, float); n, k = X.shape
    W = np.ones(n) if w is None else np.asarray(w, float)
    XtWX_inv = np.linalg.pinv(X.T @ (X * W[:, None]))          # "bread"; pinv tolerates collinearity
    beta  = XtWX_inv @ (X.T @ (W * y))
    resid = y - X @ beta
    if hc == "HC2":
        h = np.clip(W * np.einsum("ij,jk,ik->i", X, XtWX_inv, X), 0, 0.9999)   # leverages
        adj = resid**2 / (1.0 - h)
    else:  # HC1
        adj = resid**2 * (n / (n - k))
    vcov = XtWX_inv @ (X.T @ (X * (W**2 * adj)[:, None])) @ XtWX_inv            # bread·meat·bread
    return beta, vcov

def normal_ci_p(est, se, z=1.959963984540054):
    from math import erf, sqrt
    t = abs(est/se) if se > 0 else np.inf
    p = 2.0 * (1.0 - 0.5*(1.0 + erf(t/sqrt(2.0))))
    return (est - z*se, est + z*se), p

### 3c. The Lin (2013) estimator
A plain "outcome ~ treatment + controls" assumes the controls act identically in both arms. **Lin's** fix: also include `treatment × (centred control)` interactions, letting each arm have its own slopes. Centring first keeps the `treatment` coefficient interpretable as the average treatment effect. Pass `weight_col="weight"` for the **PATE**, `None` for the **SATE**.

In [ ]:
def lin_estimator(df, outcome, weight_col=None, pre_col=None, seed=RANDOM_SEED):
    d = df.dropna(subset=[outcome]).copy()
    T = d["treatment"].values.astype(float)
    Xfull = build_covariate_matrix(d, pre_col)
    keep  = select_covariates(Xfull, d[outcome].values, seed=seed)        # post-lasso controls
    stratum = pd.get_dummies(d["stratum"], prefix="stratum", drop_first=True).astype(float)
    parts = []
    if keep:
        Xc = Xfull[keep]; Xc_c = Xc - Xc.mean(axis=0)                     # centre -> Lin's trick
        inter = Xc_c.mul(T, axis=0); inter.columns = [f"T_x_{c}" for c in inter.columns]
        parts += [Xc_c, inter]
    parts.append(stratum)
    design = pd.concat(parts, axis=1)
    design.insert(0, "treatment", T); design.insert(0, "const", 1.0)
    design = design.astype(float)
    ti = list(design.columns).index("treatment")
    if weight_col is None:
        beta, vcov = ols_robust(design.values, d[outcome].values, w=None, hc="HC2")
    else:
        beta, vcov = ols_robust(design.values, d[outcome].values, w=d[weight_col].values, hc="HC1")
    est = beta[ti]; se = np.sqrt(vcov[ti, ti]); (lo, hi), p = normal_ci_p(est, se)
    return {"outcome": outcome, "estimate": est, "se": se, "ci_low": lo, "ci_high": hi,
            "p_raw": p, "n": int(len(d)), "n_controls": len(keep)}

### 3d. Sharpened FDR (Benjamini–Krieger–Yekutieli 2006)
Testing many hypotheses produces false positives by chance. **False-discovery-rate** control limits the expected share of false positives among the findings you call significant. The *sharpened* version runs Benjamini–Hochberg once to estimate how many nulls are truly null, then re-runs it with that smaller count for more power. A test is significant if its adjusted q-value `≤ 0.05`.

In [ ]:
def sharpened_fdr(pvalues, q=0.05):
    p = np.asarray(pvalues, float); m = len(p)
    order = np.argsort(p); p_sorted = p[order]
    def bh(p_sorted, m_eff):
        ranks = np.arange(1, len(p_sorted)+1)
        q_mono = np.minimum.accumulate((p_sorted*m_eff/ranks)[::-1])[::-1]   # monotone step-up
        return np.clip(q_mono, 0, 1)
    stage1 = bh(p_sorted, m)                       # stage 1: ordinary BH
    m0 = m - int(np.sum(stage1 <= q))             # estimate # of true nulls
    adj_sorted = stage1 if m0 == 0 else bh(p_sorted, m0)   # stage 2: re-run with m0
    adj = np.empty(m); adj[order] = adj_sorted
    return adj

def run_family(df, spec, pre_treatment=True, q=0.05, seed=RANDOM_SEED):
    rows = []
    for name, s in spec.items():
        pre = (name + "_pre") if pre_treatment and (name + "_pre") in df.columns else None
        pate = lin_estimator(df, name, weight_col="weight", pre_col=pre, seed=seed)
        sate = lin_estimator(df, name, weight_col=None,     pre_col=pre, seed=seed)
        rows.append({"label": s["label"], "true_effect": s.get("true_effect", np.nan),
                     "pate": pate["estimate"], "pate_se": pate["se"],
                     "pate_ci_low": pate["ci_low"], "pate_ci_high": pate["ci_high"],
                     "pate_p_raw": pate["p_raw"], "sate": sate["estimate"],
                     "sate_p_raw": sate["p_raw"], "n": pate["n"]})
    res = pd.DataFrame(rows)
    res["pate_p_adj"] = sharpened_fdr(res["pate_p_raw"].values, q=q)
    res["sate_p_adj"] = sharpened_fdr(res["sate_p_raw"].values, q=q)
    return res

## 4. First stage (the "RQ"): does the feed change?

These are auxiliary outcomes, so the paper does **not** FDR-correct them. We report the weighted PATE in percentage points. Expectation from the paper: **every one is large and significant** — the chronological feed shows you more political, more cross-cutting, and more untrustworthy content.

In [ ]:
def forest_plot(res, title, unit, signif_col="pate_p_adj"):
    res = res.iloc[::-1].reset_index(drop=True); y = np.arange(len(res))
    fig, ax = plt.subplots(figsize=(8, 0.55*len(res) + 1.5))
    for i, row in res.iterrows():
        sig = (signif_col in row) and (row[signif_col] <= 0.05)
        c = "#c0392b" if sig else "#34495e"
        ax.plot([row["pate_ci_low"], row["pate_ci_high"]], [y[i], y[i]], color=c, lw=2, zorder=2)
        ax.scatter([row["pate"]], [y[i]], color=c, s=45, zorder=3)
        if not np.isnan(row.get("true_effect", np.nan)):
            ax.scatter([row["true_effect"]], [y[i]], marker="|", color="#2980b9", s=120, zorder=4)
    ax.axvline(0, color="#999", lw=1, ls="--"); ax.set_yticks(y); ax.set_yticklabels(res["label"])
    ax.set_xlabel(f"Treatment effect (chronological feed), {unit}"); ax.set_title(title, loc="left", fontsize=11)
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([0],[0],color="#c0392b",marker="o",lw=2,label="Significant"),
                       Line2D([0],[0],color="#34495e",marker="o",lw=2,label="Not significant"),
                       Line2D([0],[0],color="#2980b9",marker="|",lw=0,markersize=12,label="Planted true effect")],
              fontsize=7, loc="lower right", frameon=False)
    plt.tight_layout(); plt.show()

rq = pd.DataFrame([{**{"label": s["label"], "true_effect": s["true_effect"]},
                    **lin_estimator(fb, name, weight_col="weight")} for name, s in RQ_OUTCOMES.items()])
rq = rq.rename(columns={"estimate":"pate","se":"pate_se","ci_low":"pate_ci_low",
                        "ci_high":"pate_ci_high","p_raw":"pate_p_raw"})
print(rq[["label","true_effect","pate","pate_se","pate_p_raw"]].to_string(index=False))
forest_plot(rq, "First stage: chronological feed changes what you see (Facebook)",
            "percentage points", signif_col="pate_p_raw")

## 5. Primary hypotheses H1–H3: the headline null

H1 = polarization, H2 = knowledge, H3 = participation/turnout/engagement. Expectation from the paper: under the main-text **PATE**, the *only* significant primary outcome is **H3c on-platform engagement** — a behaviour — while every **attitude** (H1, H2, H3a/b) is null.

Watch the **PATE vs SATE** columns: for **H2b (news knowledge)** the unweighted SATE reaches significance while the weighted PATE does not — exactly the divergence the paper reports (the weighting matters).

In [ ]:
primary = run_family(fb, PRIMARY_OUTCOMES)
print(primary[["label","true_effect","pate","pate_p_raw","pate_p_adj","sate","sate_p_raw","sate_p_adj"]]
      .to_string(index=False))
forest_plot(primary, "Primary outcomes: attitudes & behaviour (Facebook)", "SD units")

**Reading it.** The forest plot tells the whole story: six attitudinal intervals straddle zero (grey), and only H3c — a behaviour — is shifted and significant (red). A three-month switch to a chronological feed changed what people *did* on the platform but not what they *thought*.

## 6. Secondary hypotheses and what FDR does

The secondary family is where multiple-comparison correction earns its keep. Compare each outcome's raw p-value to its FDR-adjusted q-value: the correction *inflates* every p-value, and borderline raw findings can lose significance once you account for how many tests you ran.

In [ ]:
secondary = run_family(fb, SECONDARY_OUTCOMES)
print(secondary[["label","true_effect","pate","pate_p_raw","pate_p_adj"]].to_string(index=False))
forest_plot(secondary, "Secondary outcomes (Facebook)", "SD units")

## 7. Compliance: an instrumental-variables check

About 12% of treated Facebook users never actually received the chronological feed (a web bug, fixed mid-study). The paper re-estimates effects by **instrumenting the dose** (share of views in chronological order) with random assignment — a Wald/2SLS ratio that recovers the effect *among compliers* (the Local ATE). Because it divides the intention-to-treat effect by the compliance rate, the complier effect is mechanically a bit larger, but the conclusions don't change.

In [ ]:
def iv_complier_effect(df, outcome, dose="pct_views_chrono", weight_col="weight"):
    d = df.dropna(subset=[outcome]); T = d["treatment"].values
    w = d[weight_col].values; Y = d[outcome].values; D = d[dose].values
    wm = lambda a, m: np.average(a[m], weights=w[m])
    itt_y = wm(Y, T==1) - wm(Y, T==0)          # reduced form (assignment -> outcome)
    itt_d = wm(D, T==1) - wm(D, T==0)          # first stage (assignment -> dose)
    return {"itt_outcome": itt_y, "itt_dose": itt_d, "late": itt_y/itt_d}

for o in ["onplatform_engagement", "pct_untrustworthy"]:
    r = iv_complier_effect(fb, o)
    print(f"{o:24s} ITT={r['itt_outcome']:+.3f}  first-stage dose={r['itt_dose']:.3f}  "
          f"complier (LATE)={r['late']:+.3f}")

## 8. Did we recover what we planted?

The honest test of a synthetic methodological replication: the estimator should return the effects we put in. Here is recovered-vs-planted for every standardized outcome.

In [ ]:
allres = run_family(fb, ALL_STD_OUTCOMES)
allres["recovery_error"] = (allres["pate"] - allres["true_effect"]).abs()
print(allres[["label","true_effect","pate","recovery_error"]].to_string(index=False))
print(f"\nMean absolute recovery error (PATE): {allres['recovery_error'].mean():.4f} SD")
print("First-stage effects, recovered vs planted:")
print((rq[["label","true_effect","pate"]]).to_string(index=False))

## 9. Summary

* **First stage:** large, significant effects on feed composition — the intervention worked. ✔ reproduced.
* **Attitudes (H1, H2, H3a/b):** null under the weighted PATE, even though the manipulation was strong. ✔ reproduced — this is the paper's headline.
* **Behaviour (H3c):** on-platform political engagement fell. ✔ reproduced.
* **Method nuances:** PATE-vs-SATE divergence (weighting matters) and FDR inflating borderline findings both appear, as in the paper.

**What this does *not* show.** With synthetic data we cannot confirm Meta's exact magnitudes, only that the *pipeline* and the *qualitative pattern* are right. And remember the algorithm-in-flux caveat: the control "algorithmic feed" was itself changing during the study. Every divergence from the original is logged in `replication_log.md`.